## Data Walkthrough Notebook

This notebook inspects a set of 5 claims/data entries from the AVeriTeC dataset. We will examine how claims, annotated evidence and knowledge store documents relate to one another

The questions to look for an answer to:

1. What information is supplied per claim?
2. How is the claim specific knowledge store represented?
3. Does the knowledge store contain the annotated evidence source?
4. Is sentence level retrieval sufficient, or is surrounding context needed?
5. What should the initial retrieval unit be?

## 1. Setup

In [22]:
import re
import os
import json
import pandas as pd
from pathlib import Path
from IPython.display import display
from urllib.parse import parse_qsl, urlencode, urlsplit


def find_git_repository(start: Path) -> Path:
    """Find the nearest parent directory containing .git."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find a Git repository above the current directory."
    )


PROJECT_ROOT = find_git_repository(Path.cwd())

# Check the locations that may have been used by the setup notebook.
search_roots = [
    PROJECT_ROOT / ".local_data",
    PROJECT_ROOT / "notebooks" / ".local_data",
    PROJECT_ROOT / "data",
]

# Also check an explicitly configured path, when available.
configured_root = os.environ.get("AVERITEC_ROOT")

if configured_root:
    search_roots.insert(0, Path(configured_root))

dev_matches: list[Path] = []

for root in search_roots:
    if root.exists():
        dev_matches.extend(
            path
            for path in root.rglob("dev.json")
            if "averitec" in str(path).lower()
        )

# Remove duplicate paths while retaining order.
dev_matches = list(dict.fromkeys(path.resolve() for path in dev_matches))

if not dev_matches:
    locations = "\n".join(f"- {path}" for path in search_roots)

    raise FileNotFoundError(
        "No downloaded AVeriTeC dev.json file was found.\n"
        f"Searched:\n{locations}"
    )

print("Possible development annotation files:")

for index, path in enumerate(dev_matches):
    print(f"[{index}] {path}")


# ###
DEV_PATH = dev_matches[0]

# # dev.json is stored at:
# # <AVERITEC_ROOT>/data/dev.json
AVERITEC_ROOT = DEV_PATH.parent.parent

TRAIN_PATH = AVERITEC_ROOT / "data" / "train.json"
KNOWLEDGE_STORE_ROOT = AVERITEC_ROOT / "knowledge_store" / "dev"
KNOWLEDGE_STORE_ARCHIVE = (
    AVERITEC_ROOT
    / "data_store"
    / "knowledge_store"
    / "dev_knowledge_store.zip"
)

os.environ["AVERITEC_ROOT"] = str(AVERITEC_ROOT)

print("Project root:       ", PROJECT_ROOT)
print("AVeriTeC root:      ", AVERITEC_ROOT)
print("Development data:   ", DEV_PATH)
print("Training data:      ", TRAIN_PATH)
print("Knowledge store:    ", KNOWLEDGE_STORE_ROOT)

print("\nExists:")
print("dev.json:           ", DEV_PATH.exists())
print("train.json:         ", TRAIN_PATH.exists())
print("extracted store:    ", KNOWLEDGE_STORE_ROOT.exists())
print("store archive:      ", KNOWLEDGE_STORE_ARCHIVE.exists())



pd.set_option("display.max_colwidth", 250)

print("Development annotations:", DEV_PATH)
print("Knowledge store:", KNOWLEDGE_STORE_ROOT)


Possible development annotation files:
[0] C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a\data\dev.json
[1] C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data\dev.json
Project root:        C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison
AVeriTeC root:       C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a
Development data:    C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a\data\dev.json
Training data:       C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\

## 1.1. Inspect a single claim

In [23]:
with DEV_PATH.open("r", encoding="utf-8") as file:
    dev_claims = json.load(file)

if not isinstance(dev_claims, list):
    raise TypeError(
        f"Expected the development data to contain a list, "
        f"but received {type(dev_claims).__name__}."
    )

print(f"Loaded {len(dev_claims):,} development claims.")

CLAIM_ID = 0

if not 0 <= CLAIM_ID < len(dev_claims):
    raise IndexError(
        f"CLAIM_ID must be between 0 and {len(dev_claims) - 1}."
    )

claim = dev_claims[CLAIM_ID]

print("CLAIM")
print("-----")
print(claim.get("claim", "[missing claim]"))

print("\nLABEL")
print("-----")
print(claim.get("label", "[missing label]"))

print("\nJUSTIFICATION")
print("-------------")
print(claim.get("justification", "[missing justification]"))

Loaded 500 development claims.
CLAIM
-----
In a letter to Steve Jobs, Sean Connery refused to appear in an apple commercial.

LABEL
-----
Refuted

JUSTIFICATION
-------------
The answer and sources show that the claim was published in a fake news site so the claim is refuted.


In [24]:
gold_rows = []

for question_number, question in enumerate(claim.get("questions") or [], start=1):
    answers = question.get("answers") or []

    if not answers:
        gold_rows.append({"question_number": question_number, "answer_number": None, "question": question.get("question"), "answer": None, "answer_type": None, "source_url": None, "cached_source_url": None})
        continue

    for answer_number, answer in enumerate(answers, start=1):
        gold_rows.append({"question_number": question_number, "answer_number": answer_number, "question": question.get("question"), "answer": answer.get("answer"), "answer_type": answer.get("answer_type"), "source_url": answer.get("source_url"), "cached_source_url": answer.get("cached_source_url")})

gold_evidence = pd.DataFrame(gold_rows)

print(f"Annotated questions: " f"{gold_evidence['question_number'].nunique():,}")
print(f"Annotated answers: {gold_evidence['answer'].notna().sum():,}")

display(gold_evidence)

Annotated questions: 2
Annotated answers: 2


,question_number,answer_number,question,answer,answer_type,source_url,cached_source_url
0,1,1,Where was the claim first published,It was first published on Sccopertino,Abstractive,https://web.archive.org/web/20201129141238/https://scoopertino.com/exposed-the-imac-disaster-that-almost-was/,https://web.archive.org/web/20201129141238/https://scoopertino.com/exposed-the-imac-disaster-that-almost-was/
1,2,1,What kind of website is Scoopertino,"Scoopertino is an imaginary news organization devoted to ferreting out the most relevant stories in the world of Apple, whether or not they actually occurred - says their about page",Extractive,https://web.archive.org/web/20201202085933/https://scoopertino.com/about-scoopertino/,https://web.archive.org/web/20201202085933/https://scoopertino.com/about-scoopertino/


## 2. Load and show the claim’s knowledge store

In [25]:
def resolve_claim_store_path(claim_id: int, root: Path) -> Path:
    """Locate the knowledge-store file associated with one claim."""

    root = Path(root)

    direct_candidates = [root / f"{claim_id}.json", root / f"{claim_id}.jsonl"]

    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    recursive_matches = [*root.rglob(f"{claim_id}.json"), *root.rglob(f"{claim_id}.jsonl")]

    recursive_matches = list(dict.fromkeys(path.resolve() for path in recursive_matches))

    if len(recursive_matches) == 1:
        return recursive_matches[0]

    if len(recursive_matches) > 1:
        raise RuntimeError(f"Multiple knowledge-store files were found for " f"claim {claim_id}:\n" + "\n".join(str(path) for path in recursive_matches))

    raise FileNotFoundError(f"No knowledge-store file for claim {claim_id} " f"was found under:\n{root}")

In [26]:
def read_knowledge_store_records(path: Path) -> list[dict]:
    """
    Read a claim knowledge-store file.

    Supports both:
    - one JSON object per line;
    - a complete JSON list or object.
    """

    raw_text = path.read_text(encoding="utf-8").strip()

    if not raw_text:
        return []

    # First try reading the entire file as ordinary JSON.
    try:
        parsed = json.loads(raw_text)

        if isinstance(parsed, list):
            return parsed

        if isinstance(parsed, dict):
            return [parsed]

        raise TypeError(f"Unexpected JSON type: {type(parsed).__name__}")

    except json.JSONDecodeError:
        pass

    # Otherwise interpret it as JSON Lines.
    records = []

    for line_number, line in enumerate(raw_text.splitlines(), start=1):
        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON on line {line_number} of {path}.") from exc

        if isinstance(record, dict):
            records.append(record)

    return records

In [27]:
def load_claim_knowledge_store(claim_id: int, root: Path) -> pd.DataFrame:
    """Load and flatten one claim's knowledge-store sources."""

    path = resolve_claim_store_path(claim_id, root)
    source_records = read_knowledge_store_records(path)

    rows = []

    for source_number, source in enumerate(source_records):
        url = source.get("url")

        text_units = (source.get("url2text") or source.get("text") or [])

        if isinstance(text_units, str):
            text_units = [text_units]

        if not isinstance(text_units, list):
            continue

        for sentence_number, sentence in enumerate(text_units):
            if not isinstance(sentence, str):
                continue

            sentence = sentence.strip()

            if not sentence:
                continue

            rows.append({"claim_id": claim_id, "source_number": source_number, "source_id": f"{claim_id}:{source_number}", "sentence_number": sentence_number, "sentence_id": (f"{claim_id}:{source_number}:{sentence_number}"), "url": url, "text": sentence})

    result = pd.DataFrame(rows)

    if result.empty:
        raise ValueError(f"No usable source sentences were found for claim " f"{claim_id} in {path}.")

    return result

In [28]:
claim_store_path = resolve_claim_store_path(CLAIM_ID, KNOWLEDGE_STORE_ROOT)

candidate_sentences = load_claim_knowledge_store(CLAIM_ID, KNOWLEDGE_STORE_ROOT)

print("Knowledge-store file:")
print(claim_store_path)

print("\nCollection statistics")
print("---------------------")
print(f"Source records: " f"{candidate_sentences['source_id'].nunique():,}")
print(f"Unique URLs: " f"{candidate_sentences['url'].nunique():,}")
print(f"Extracted sentences: {len(candidate_sentences):,}")

display(candidate_sentences.head(20))

FileNotFoundError: No knowledge-store file for claim 0 was found under:
C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a\knowledge_store\dev

## 3.1 Inspect sentences from gold source

In [ ]:
gold_url = gold_evidence.loc[gold_evidence["exact_url_in_store"],"source_url"].iloc[0]
gold_source_sentences = candidate_sentences[candidate_sentences["url"] == gold_url].copy()

pd.set_option("display.max_colwidth", 200)
gold_source_sentences[["sentence_number", "text"]].head(30)

## 4. Compare candidate retrieval units

In [ ]:
sentences = gold_source_sentences["text"].tolist()

sentence_units = sentences

three_sentence_passages = [" ".join(sentences[i : i + 3]) for i in range(0, len(sentences), 2)]
five_sentence_passages = [" ".join(sentences[i : i + 5]) for i in range(0, len(sentences), 3)]